In [ ]:
import glob
import json
import os
from collections import defaultdict
import numpy as np
import pandas as pd
import torch

JSON_GLOB = "/scratch1/eibl/data/covid19_twitter/raw/*/*.json"

pd.set_option('display.max_columns', None)

In [ ]:
def load_json_items(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read().strip()
    if not text:
        return []
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
        if isinstance(obj, dict):
            if isinstance(obj.get("statuses"), list):
                return obj["statuses"]
            if isinstance(obj.get("data"), list):
                return obj["data"]
            return [obj]
        return []
    except json.JSONDecodeError:
        items = []
        for line in text.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                items.append(json.loads(line))
            except json.JSONDecodeError:
                pass
        return items


def parse_tweets(path):
    rows = []
    try:
        items = load_json_items(path)
    except Exception as e:
        print(f"Skipping {path}: {e}")
        return rows
    for tweet in items:
        user = tweet.get("user") or {}
        rt = tweet.get("retweeted_status") or {}
        rt_user = rt.get("user") or {}
        entities = tweet.get("entities") or {}
        urls = entities.get("urls", []) or []
        # display_url looks like "domain.com/path" — split on '/' to get domain
        display_urls = [u.get("display_url", "") for u in urls if isinstance(u, dict)]
        rows.append({
            "userid": user.get("id"),
            "screen_name": user.get("screen_name"),
            "description": user.get("description"),
            "rt_screen": rt_user.get("screen_name") if rt else None,
            "display_urls": display_urls,
        })
    return rows


files = sorted(glob.glob(JSON_GLOB))
print(f"Found {len(files)} JSON files")

chunks = []
skipped = 0
for i, f in enumerate(files):
    try:
        print(f"({i}) reading {os.path.basename(f)}", flush=True, end='\r')
        rows = parse_tweets(f)
        if rows:
            chunks.append(pd.DataFrame(rows))
        else:
            skipped += 1
    except Exception as e:
        print(f"Skipping {f}: {e}")
        skipped += 1

df = pd.concat(chunks, ignore_index=True)
df['userid'] = pd.to_numeric(df['userid'], errors='coerce')
df = df.dropna(subset=['userid']).copy()
df['userid'] = df['userid'].astype(int)
print(f"Skipped files: {skipped}")
print(f"Total rows: {len(df):,}  |  Unique users: {df['userid'].nunique():,}")

## mappers

In [ ]:
domain2score = {
    'abcnews.go.com': 2.0,
     'bbc.com': 3.0,
     'breitbart.com': 5.0,
     'bostonglobe.com': 2.0,
     'businessinsider.com': 3.0,
     'buzzfeednews.com': 1.0,
     'cbsnews.com': 2.0,
     'chicagotribune.com': 3.0,
     'cnbc.com': 3.0,
     'cnn.com': 2.0,
     'dailycaller.com': 5.0,
     'dailymail.co.uk': 5.0,
     'foxnews.com': 4.0,
     'huffpost.com': 1.0,
     'infowars.com': 5.0,
     'latimes.com': 2.0,
     'msnbc.om': 1.0,
     'nbcnews.com': 2.0,
     'nytimes.com': 2.0,
     'npr.org': 3.0,
     'oann.com': 4.0,
     'pbs.org': 3.0,
     'reuters.com': 3.0,
     'theguardian.com': 2.0,
     'usatoday.com': 3.0,
     'yahoo.com': 2.0,
     'vice.com': 1.0,
     'washingtonpost.com': 2.0,
     'wsj.com': 3.0,
      
    # additional
     'thehill.com': 2.7,
     'rt.com': 3.7,
     'rawstory.com': 1.7,
     'news.sky.com': 2.3,
     'independent.co.uk': 2.3,
     'dailykos.com': 1.7
}

In [ ]:
domain2canonical = {
    
    'bit.ly': None,  # URL shortener
    'dlvr.it': None,  # syndication tool
    'trib.al': None,  # URL shortener
    'ift.tt': None,  # IFTTT shortener
    'twibbon.com': None,  # campaign tool
    't.me': None,  # Telegram shortener
    'apple.news': None,  # aggregator
    'ow.ly': None,  # Hootsuite shortener
    'buff.ly': None,  # Buffer shortener
    'tinyurl.com': None,  # URL shortener
    'news.google.com': None,  # aggregator
    'lnr.app': None,  # URL shortener
    'fiverr.com': None,  # freelancing platform, likely data artifact
    
    'twitter.com': 'twitter.com',
    'reut.rs': 'reuters.com',
    'youtu.be': 'youtube.com',
    'theguardian.com': 'theguardian.com',
    'youtube.com': 'youtube.com',
    'nytimes.com': 'nytimes.com',
    'reuters.com': 'reuters.com',
    'washingtonpost.com': 'washingtonpost.com',
    'cnn.com': 'cnn.com',
    'businessinsider.com': 'businessinsider.com',
    'rt.com': 'rt.com',
    'hill.cm': 'thehill.com',
    'bbc.in': 'bbc.com',
    'foxnews.com': 'foxnews.com',
    'facebook.com': 'facebook.com',
    'rawstory.com': 'rawstory.com',
    'bbc.com': 'bbc.com',
    'bbc.co.uk': 'bbc.com',
    'news.sky.com': 'news.sky.com',
    'a.msn.com': 'msn.com',
    'npr.org': 'npr.org',
    'politico.com': 'politico.com',
    'en.wikipedia.org': 'wikipedia.org',
    'cnb.cx': 'cnbc.com',
    'timesofindia.indiatimes.com': 'timesofindia.indiatimes.com',
    'dailykos.com': 'dailykos.com',
    'independent.co.uk': 'independent.co.uk',
    'msn.com': 'msn.com',
    'jp.reuters.com': 'reuters.com',
    'axios.com': 'axios.com',
    'news.yahoo.com': 'yahoo.com',
    'thehill.com': 'thehill.com',
    'cnbc.com': 'cnbc.com',
    'bloomberg.com': 'bloomberg.com',
    'wsj.com': 'wsj.com',
    'theatlantic.com': 'theatlantic.com',
    'on.ft.com': 'ft.com',
    'cnn.it': 'cnn.com',
    'whitehouse.gov': 'whitehouse.gov',
    'nbcnews.com': 'nbcnews.com',
    'nypost.com': 'nypost.com',
    'ft.com': 'ft.com',
    'wapo.st': 'washingtonpost.com',
    'aljazeera.com': 'aljazeera.com',
    'who.int': 'who.int',
    'sputniknews.com': 'sputniknews.com',
    'mol.im': 'dailymail.co.uk',
    'washingtontimes.com': 'washingtontimes.com',
}

In [ ]:
handle2score = {
    'abc': 2,
 'bbcworld': 3,
 'breitbartnews': 5,
 'bostonglobe': 2,
 'businessinsider': 3,
 'buzzfeednews': 1,
 'cbsnews': 2,
 'chicagotribune': 3,
 'cnbc': 3,
 'cnn': 2,
 'dailycaller': 5,
 'dailymail': 5,
 'foxnews': 4,
 'huffpost': 1,
 'infowars*': 5,
 'latimes': 2,
 'msnbc': 1,
 'nbcnews': 2,
 'nytimes': 2,
 'npr': 3,
 'oann': 4,
 'pbs': 3,
 'reuters': 3,
 'guardian': 2,
 'usatoday': 3,
 'yahoonews': 2,
 'vice': 1,
 'washingtonpost': 2,
 'wsj': 3,
 # additional
 'thehill': 2.7,
 'rt_com': 3.7,
 'rawstory': 1.7,
 'skynews': 2.3,
 'independent': 2.3,
 'dailykos': 1.7
}

In [ ]:
hashtag2score = {'blm': 0,'resist': 0,'fbpe': 0,'blacklivesmatter': 0,'fbr': 0,'maga': 1,'theresistance': 0,'voteblue': 0,'resistance': 0,'bidenharris': 0,'johnsonout': 0,'lgbtq': 0,'bidenharris2020': 0,'2a': 1,'fbpa': 0,'resister': 0,'fbppr': 0,'bluewave': 0,'gtto': 0,'freepalestine': 0,'rejoineu': 0,'voteblue2022': 0,'kag': 1,'wearamask': 0,'getvaccinated': 0,'humanrights': 0,'votingrights': 0,'science': 0,'goodtrouble': 0,'strongertogether': 0,'stillwithher': 0,'climatecrisis': 0,'metoo': 0,'demvoice1': 0,'biden': 0,'climatechange': 0,'justicematters': 0,'americafirst': 1,'nevertrump': 0,'khive': 0,'democrat': 0,'vaccinated': 0,'buildbackbetter': 0,'stopasianhate': 0,'prochoice': 0,'drcole': 1,'bds': 0,'votebluenomatterwho': 0,'teampelosi': 0,'handmarkedpaperballots': 1,'reconquête': 1,'biden2020': 0,'patriot': 1,'equality': 0,'prolife': 1,'antifa': 0,'fjb': 1,'lgbt': 0,'nra': 1,'climateaction': 0,'unionpopulaire': 0,'zemmour2022': 1,'trump': 1,'demcast': 0,'m4a': 0,'vaxxed': 0,'democracy': 0,'indivisible': 0,'teamjustice': 0,'noafd': 0,'alllivesmatter': 1}

In [ ]:
## URL
# display_url from JSON entities looks like "domain.com/path..." — split on '/' to get domain
def get_domain(display_url):
    return display_url.split('/')[0] if display_url else ''

url_scores = (
    df['display_urls']
    .apply(lambda urls: [get_domain(u) for u in urls] if isinstance(urls, list) else [])
    .apply(lambda domains: [domain2canonical.get(d, d) for d in domains])
    .apply(lambda domains: [domain2score.get(d) for d in domains if d in domain2score])
)
url_scores = url_scores[url_scores.str.len().gt(0)]
df['url_scores'] = url_scores
mask = df.url_scores.notna()
df['mean_tweet_url_pol_score'] = np.nan
mean_tweet_url_pol_score = df.loc[mask, 'url_scores'].apply(
    lambda x: np.nanmean([0 if y < 3 else 1 if y > 3 else np.nan for y in x])
)
mean_tweet_url_pol_score = mean_tweet_url_pol_score[~mean_tweet_url_pol_score.between(1/5, 4/5)]
df['mean_tweet_url_pol_score'] = mean_tweet_url_pol_score
user2urlScore = pd.crosstab(df['userid'], df['mean_tweet_url_pol_score'])
user2urlScore.columns = ['url_left', 'url_right']

## RETWEET
rt_org_pol_score = df.rt_screen.str.lower().map(handle2score)
df['rt_org_pol_score'] = np.nan
i = rt_org_pol_score.gt(3)
df.loc[i, 'rt_org_pol_score'] = 'rt_right'
j = rt_org_pol_score.lt(3)
df.loc[j, 'rt_org_pol_score'] = 'rt_left'
user2rtScore = pd.crosstab(df['userid'], df['rt_org_pol_score'])

## HASHTAG (from user bio/description)
df['description'] = df['description'].fillna('')
df["descr_hashtags"] = df.drop_duplicates(subset=['userid', 'description']).description.str.lower().str.findall(r'\#(\w+)')
i = df.descr_hashtags.str.len().lt(1)
df.loc[i, 'descr_hashtags'] = np.nan
hashtag_scores = (
    df['descr_hashtags'].dropna()
    .apply(lambda x: [hashtag2score.get(u) for u in x if u in hashtag2score])
)
hashtag_scores = hashtag_scores[hashtag_scores.str.len().gt(0)]
hashtag_scores = hashtag_scores.apply(lambda x: pd.Series(x).value_counts())
df[['hashtag_left', 'hashtag_right']] = hashtag_scores
user2hashtagScore = df.groupby('userid')[['hashtag_left', 'hashtag_right']].sum()

## USER2SCORE
user2score = pd.concat([user2urlScore, user2rtScore, user2hashtagScore], axis=1, join='outer')
left_cols = ['url_left', 'rt_left', 'hashtag_left']
right_cols = ['url_right', 'rt_right', 'hashtag_right']
user2score['left_vote'] = user2score[left_cols].sum(1)
user2score['right_vote'] = user2score[right_cols].sum(1)

user2pol = user2score.right_vote.gt(0) & user2score.left_vote.gt(0)
user2pol = user2pol * 0 - 1
right_mask = user2score.right_vote.gt(1) & user2score.left_vote.eq(0)
user2pol[right_mask] = 1
left_mask = user2score.right_vote.eq(0) & user2score.left_vote.gt(1)
user2pol[left_mask] = 0
user2pol.value_counts()

In [ ]:
d = user2score[['left_vote', 'right_vote']].gt(1)
d[d.sum(1).gt(1)] = False

In [ ]:
d.value_counts(normalize=True)